# United States: Okun's law and the wage Phillips curve

Quarterly change in unemployment against real GDP growth (Okun), and nominal wage growth against unemployment (Phillips).

**Datasets:** `bls/headline_monthly`, `fred/distribution_and_profits`, `fred/labour_market_extras`

This notebook reads the latest vintage each time it runs; re-run after a new release to update every table and chart.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from labobs.load import load, vintage, curated, raw_payload
from labobs.nb import select, pivot, latest, plot_lines, plot_bars, to_annual, FOCUS, FOCUS_NAMES
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
DATASETS = ["bls/headline_monthly", "fred/distribution_and_profits", "fred/labour_market_extras"]
for k in DATASETS:
    v = vintage(k)
    print(f"{k:50} vintage {v.get('vintage', 'not yet ingested'):12} sha {v.get('sha256', '')[:12]}")

In [ ]:
import numpy as np
ur = to_annual(select(load("bls/headline_monthly"), indicator="LNS14000000"))
gdp = select(load("fred/distribution_and_profits"), indicator="GDP").copy()
gdp["period"] = gdp["period"].astype(str).str[:4]
g = gdp.groupby("period")["value"].mean().pct_change() * 100
u = ur.set_index("period")["value"].diff()
ok = pd.concat({"gdp_growth_nominal": g, "d_unemployment": u}, axis=1).dropna()
b = np.polyfit(ok.gdp_growth_nominal, ok.d_unemployment, 1)
print(f"Okun slope (nominal GDP, annual): {b[0]:.2f}  — replace GDP with real GDP (GDPC1) for the textbook coefficient")
ok.tail()

In [ ]:
wage = select(load("fred/labour_market_extras"), indicator="LES1252881600Q").copy()
wage["period"] = wage["period"].astype(str).str[:4]
wg = wage.groupby("period")["value"].mean().pct_change() * 100
ph = pd.concat({"real_median_weekly_earnings_growth": wg, "unemployment": ur.set_index("period")["value"]}, axis=1).dropna()
ax = ph.plot.scatter(x="unemployment", y="real_median_weekly_earnings_growth", figsize=(7, 5))
ax.set_title("Real earnings growth vs unemployment", loc="left");

## Curator notes

_Written by the curator after each new vintage: what changed, which revisions matter, and what to watch. See `DATA_CHANGELOG.md` for the full log._